In [ ]:
<a href="https://colab.research.google.com/github/vanderbilt-data-science/MNPSCollaborative/blob/Fixed-Two-Pass-v7.5.5/MNPS_Job_Classification_GPT4o_Two_Pass_Fixed_v7.5.5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
# **MNPS Job Classification GPT-4o (Two-Pass + Error Fixes v7.5.5)**
> A notebook to help you get started  
> DSI DSSG + MNPS   
> # **Version 7.5.5 Changes**
> - **Fixed Coordinator/Manager misclassification** (Rows 2, 26, 29)
> - **Added Specialist override for licensed O&M roles** (Row 19)
> - **Normalized minor sub-group to blank for Teacher/Therapist/Counselor/Principal/Librarian**
> - **Added “Representative” role** for low-complexity coordinators (Row 34)
> - **Promote to “Director” for large-scale facility leadership** (Row 37)
> - **Downgrade Coordinator III unless senior executive indicators present** (Row 32)
> - **Post-processing applied AFTER Pass 2** to preserve self-consistency
> - **Two-Pass logic preserved**: Pass 1 → Pass 2 → Post-processing → Output
> - **GPT-4o-2024-11-20 model retained**
> - **All MNPS prompts and validation intact**

In [ ]:
# ==== 1) Imports, paths, inputs ====
import os, json, shutil, datetime as dt, zipfile
from pathlib import Path
import pandas as pd
import numpy as np
import re
import time
import random
from google.colab import drive
from openai import OpenAI

# Mount Google Drive
drive.mount('/content/drive')

# Create unique run folder
timestamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
run_folder = f"RUN_{timestamp}"
base_path = Path("/content/drive/My Drive/Colab Notebooks/Run Results")
run_path = base_path / run_folder
run_path.mkdir(parents=True, exist_ok=True)

# Create outputs subfolder
OUTPUTS_DIR = run_path / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"📁 Run folder: {run_path}")
print(f"📁 Outputs dir: {OUTPUTS_DIR}")

# Load all required files
RUN_ROOT = Path('/content')
ZIP_FILE = RUN_ROOT / "MNPS Prompt Resources.zip"

if ZIP_FILE.exists():
    print(f"📦 Found {ZIP_FILE}, extracting...")
    with zipfile.ZipFile(ZIP_FILE, 'r') as zip_ref:
        zip_ref.extractall(RUN_ROOT)
    print("✅ Extracted MNPS Prompt Resources")
else:
    print("⚠️  MNPS Prompt Resources.zip not found")

# Core data files — check inside zip if not in root
BATCH_INPUT_CSV = RUN_ROOT / "Sample JDs.csv"
GT_MASTERFILE_CSV = RUN_ROOT / "Ground Truth Masterfile.csv"

if not BATCH_INPUT_CSV.exists() and ZIP_FILE.exists():
    print("⚠️  Sample JDs.csv not found in root — checking inside zip...")
    with zipfile.ZipFile(ZIP_FILE, 'r') as zip_ref:
        zip_contents = zip_ref.namelist()
        if "Sample JDs.csv" in zip_contents:
            zip_ref.extract("Sample JDs.csv", RUN_ROOT)
            print("✅ Extracted Sample JDs.csv from zip")
        else:
            raise FileNotFoundError("Sample JDs.csv not in root or zip")
    if "Ground Truth Masterfile.csv" in zip_contents:
        with zipfile.ZipFile(ZIP_FILE, 'r') as zip_ref:
            zip_ref.extract("Ground Truth Masterfile.csv", RUN_ROOT)

# MNPS resources
MNPS_ROLES_CSV = RUN_ROOT / "MNPS Roles.csv"
MNPS_KSACS_CSV = RUN_ROOT / "MNPS KSACs.csv"
COMPETENCY_EXTENDED_CSV = RUN_ROOT / "Competency Extended Descriptions.csv"
KORN_FERRY_CSV = RUN_ROOT / "Korn_Ferry Lominger 38 Competencies.csv"

# Load data
df = pd.read_csv(BATCH_INPUT_CSV, encoding='latin1')
roles_df = pd.read_csv(MNPS_ROLES_CSV, encoding='latin1')
ksacs_df = pd.read_csv(MNPS_KSACS_CSV, encoding='latin1')
competency_df = pd.read_csv(COMPETENCY_EXTENDED_CSV, encoding='latin1')
korn_ferry_df = pd.read_csv(KORN_FERRY_CSV, encoding='latin1')

print(f"✅ Loaded {len(df)} job descriptions")

In [ ]:
# ==== 2) Attribute-only view (ignore title) ====
ATTR_COLS = ['Position Summary', 'Essential Functions', 'Work Experience', 'Education',
             'Licenses and Certifications', 'Knowledge, Skills and Abilities']
attrs = df[ATTR_COLS].fillna('')
text = (attrs['Position Summary'] + ' ' + attrs['Essential Functions'] + ' ' +
        attrs['Work Experience'] + ' ' + attrs['Education'] + ' ' +
        attrs['Licenses and Certifications'] + ' ' + attrs['Knowledge, Skills and Abilities'])
print(f"✅ Built attribute-only view")

In [ ]:
# ==== 3) Closed sets and helpers ====
role_columns = [col for col in roles_df.columns if 'role' in col.lower()]
VALID_ROLES = roles_df[role_columns[0]].dropna().tolist() if role_columns else roles_df.iloc[:, 0].dropna().tolist()

# Updated to include "Representative" and "Liaison"
MAJOR_ALLOWED = [
    'Technician', 'Specialist', 'Analyst', 'Manager', 'Coordinator', 'Director', 'Other',
    'Teacher', 'Coach', 'Counselor', 'Clerical Support', 'Instructor', 'Driver',
    'Supervisor', 'Accountant', 'Architect (Facility-Focused)', 'Architect (Technology-Focused)',
    'Principal', 'Librarian', 'Social Worker', 'Therapist', 'Translator', 'Skilled Laborer',
    'Administrative Assistant', 'Liaison', 'Representative'
]
MINOR_ALLOWED = ['I', 'II', 'III', 'Lead']
EXECUTIVE_ROLES = ['Coordinator', 'Principal', 'Director', 'Manager']
NO_MINOR_ROLES = {'Teacher', 'Counselor', 'Principal', 'Librarian', 'Therapist', 'Assistant Principal'}

CANON_MINOR_MAP = {
    'i': 'I', '1': 'I', 'ii': 'II', '2': 'II', 'iii': 'III', '3': 'III', 'lead': 'Lead'
}

SPECIALIST_FALLBACKS = [
    ('Technician', 'technical|repair|maintenance|install|troubleshoot|equipment|hands-on|tools|machinery|systems'),
    ('Analyst', 'analyze|data analysis|research|evaluate|assess|statistical|quantitative|qualitative|metrics|reports'),
    ('Teacher', 'classroom|lesson|instruction|teacher|students|curriculum|teaching|educational|academic'),
    ('Coach', 'coach|instructional coach|plc|model lessons|co-teach|mentor|professional development|instructional support'),
    ('Clerical Support', 'clerk|clerical|records|data entry|office support|administrative|filing|correspondence'),
    ('Counselor', 'counsel|social-emotional|guidance|therapy|mental health|behavioral|psychological'),
    ('Manager', 'manage|supervise|budget|oversight|lead team|program manager|direct|strategic|planning|policy'),
    ('Accountant', 'accounting|financial|bookkeeping|audit|budget|finance|accounts payable|accounts receivable|fiscal'),
    ('Coordinator', 'coordinate|organize|facilitate|liaison|program coordination|project coordination|event coordination'),
    ('Architect (Facility-Focused)', 'building|construction|facility|architectural|design|space planning|renovation|infrastructure'),
    ('Architect (Technology-Focused)', 'system|software|technology|IT|database|network|programming|technical architecture')
]

def normalize_minor(x):
    if pd.isna(x): return 'I'
    s = str(x).strip()
    if s in MINOR_ALLOWED: return s
    return CANON_MINOR_MAP.get(s.lower(), 'I')

def discourage_specialist(text, proposed_major):
    if proposed_major != 'Specialist': return proposed_major
    t = (text or '').lower()
    for major, pattern in SPECIALIST_FALLBACKS:
        if re.search(pattern, t):
            return major
    return proposed_major

def override_for_specialist_license(text, title, proposed_major):
    if proposed_major in ['Instructor', 'Teacher']:
        license_text = (text or '').lower()
        title_lower = (title or '').lower()
        if any(kw in license_text or kw in title_lower for kw in ['orientation and mobility', 'o&m license', 'board certified']):
            return 'Specialist'
    return proposed_major

def downgrade_to_representative(text, title, proposed_major):
    if proposed_major == 'Coordinator':
        t = (text or '').lower()
        if ('high school' in t or 'ged' in t) and re.search(r'(custodial|minor repairs|clerical support|light maintenance)', t):
            return 'Representative'
    return proposed_major

def promote_to_director_if_qualified(text, proposed_major):
    if proposed_major == 'Manager':
        t = (text or '').lower()
        if ('200+ employees' in t or 'large workforce' in t) and ('pe license' in t or 'professional engineer' in t or 'architect license' in t):
            return 'Director'
    return proposed_major

def refine_coordinator_level(text, major_role, minor_role):
    if major_role == 'Coordinator' and minor_role == 'III':
        t = (text or '').lower()
        if not any(ind in t for ind in ['cabinet-level', 'district-wide budget', 'senior leadership team', 'reports to executive']):
            return 'II'
    return minor_role

def refine_coordinator_coach_manager(text, proposed_major):
    if proposed_major not in ['Coordinator', 'Coach', 'Manager']:
        return proposed_major
    t = (text or '').lower()
    
    # NEW: Demote Manager → Coordinator if no supervision or budget authority
    if proposed_major == 'Manager':
        has_supervision = re.search(r'(supervise|direct report|performance review|hire|fire|terminate|staff management)', t)
        has_strategic_budget = re.search(r'(\$\d{4,}|budget oversight|fiscal authority|appropriations)', t)
        if not (has_supervision or has_strategic_budget):
            if re.search(r'(coordinate|liaison|facilitate|program support|stakeholder engagement)', t):
                return 'Coordinator'
    
    if re.search(r'(instructional|mentor|professional development|co-teach|plc)', t):
        return 'Coach'
    if re.search(r'(strategic|policy|budget|supervise|manage|oversight|planning)', t):
        return 'Manager'
    if re.search(r'(coordinate|organize|facilitate|liaison|program|project)', t):
        return 'Coordinator'
    return proposed_major

def distinguish_supervisor_manager(text, proposed_major):
    if proposed_major not in ['Supervisor', 'Manager']: return proposed_major
    t = (text or '').lower()
    has_degree = re.search(r'(associate|bachelor|master|degree|college)', t)
    has_broader = re.search(r'(budget|strategic|policy|program|project|planning|analysis)', t)
    if has_degree or has_broader: return 'Manager'
    if re.search(r'(supervise|oversee|direct|lead team|staff management)', t): return 'Supervisor'
    return proposed_major

def fix_executive_minor_sub_grouping(major_role, minor_role):
    if major_role not in EXECUTIVE_ROLES: return minor_role
    if minor_role == 'Lead':
        return 'III' if major_role in ['Director', 'Principal'] else 'II'
    return minor_role

def finalize_minor_role(major_role, minor_role):
    if major_role in NO_MINOR_ROLES:
        return ''  # Blank per MNPS convention
    return minor_role

print("✅ Closed sets and helpers defined with all fixes")

In [ ]:
# ==== 4) Build KSACs text ====
def build_ksacs_text():
    ksacs_text = "MNPS Knowledge, Skills, Abilities, and Competencies (KSACs):\n"
    for df_obj, role_key, ksac_key in [(ksacs_df, 'role', 'ksacs'), (competency_df, 'competency', 'description'), (korn_ferry_df, 'competency', 'definition')]:
        df_obj.columns = df_obj.columns.str.strip()
        role_col = next((col for col in df_obj.columns if role_key in col.lower()), None)
        ksac_col = next((col for col in df_obj.columns if ksac_key in col.lower()), None)
        if role_col and ksac_col:
            for _, row in df_obj.iterrows():
                r = row.get(role_col, ''); k = row.get(ksac_col, '')
                if r and k:
                    ksacs_text += f"**{r}**:\n{k}\n"
    return ksacs_text

KSACS_TEXT = build_ksacs_text()
print(f"✅ Built KSACs text ({len(KSACS_TEXT)} characters)")

In [ ]:
# ==== 5) Prompts ====
zero_shot_prompt = \
""" Objective: Evaluate and group jobs from the "New Sample_08.07.2025.csv" file based on similarities in job functions, not job titles.
Process:
- Compare all jobs against each other using the attributes listed in the file: Education, Work Experience, Licenses/Certifications, Essential Functions, Knowledge, Skills, Abilities, and Position Summary.
- Group jobs that have similar functions, responsibilities, and requirements, regardless of their job titles.
- Use the attached reference sources (Ground Truth Masterfile, MNPS Roles, MNPS KSACs) to ensure alignment with MNPS standards and classifications.
- Focus on the actual work being performed, not the job title, to create meaningful and accurate groupings.
- Ensure that each grouping reflects the true nature of the work and aligns with MNPS role classifications and competency frameworks.
- Provide clear justification for each grouping decision based on the job attributes and MNPS standards.
- Never justify classifications based on job titles - only use job attributes and MNPS standards.
IMPORTANT CLASSIFICATION GUIDELINES (Based on Problem Role Cheat Sheet):
ROLE DISTINCTIONS:
- **Technician vs Specialist vs Analyst**:
  * Technician: Hands-on technical work, equipment maintenance, repair, installation, troubleshooting
  * Specialist: Specialized knowledge in specific domain, but prefer more specific roles when possible
  * Analyst: Data analysis, research, evaluation, assessment, statistical work, reporting
- **Coordinator vs Coach vs Manager**:
  * Coordinator: Coordination, organization, facilitation, liaison work, program coordination
  * Coach: Instructional support, mentoring, professional development, co-teaching, PLC facilitation
  * Manager: Strategic planning, policy development, budget oversight, supervision, management
- **Supervisor vs Manager**: 
  * Supervisor: Primarily manages people, no post-high school education required
  * Manager: Does more than manage people, requires minimum associates degree
- **Architect Roles**:
  * Architect (Facility-Focused): Building/construction/space planning/renovation/infrastructure
  * Architect (Technology-Focused): System/software/IT/database/network/programming
MINOR SUB-GROUP GUIDELINES:
- **Executive Roles** (Coordinator, Principal, Director, Manager): Rarely "Lead", usually "I", "II", or "III"
- **"Lead"** should be reserved for non-executive roles that lead teams or projects
- **"III"** for very advanced KSACs and senior-level expertise
- **"II"** for intermediate complexity and responsibility
- **"I"** for entry-level or basic complexity
Output Requirements:
- Major Role Group: Choose from approved MNPS major role groupings
- Minor Sub Group: Use I, II, III, or Lead based on complexity and responsibility level (consider executive role guidelines)
- new_job_title: Should incorporate both major_role_group and minor_sub_group (e.g., "Accountant II", "Facility Coordinator II")
- Provide detailed justification based on job attributes and MNPS KSACs alignment that matches your selected role and level"""

self_consistency_prompt = \
"""You are performing a STRICT self-consistency audit of your own prior output.
**RULES:**
1. Read ONLY your `grouping_justification`.
2. Determine what role is **explicitly described** in that justification.
3. Compare it to your stated `major_role_group`.
4. IF THEY DIFFER → **you must change `major_role_group`, `minor_sub_group`, and `new_job_title` to match the justification.**
5. IF THEY MATCH → return unchanged.
**You are NOT allowed to:**
- Keep a classification that contradicts your own reasoning
- Assume the initial prediction was correct
- Modify the justification
**Output must be valid JSON with the exact fields.**"""

print("✅ Prompts defined")

In [ ]:
# ==== 6) OpenAI API Setup ====
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
client = OpenAI()
MODEL_ID = "gpt-4o-2024-11-20"

def call_llm_json_with_retry(prompt, model=None, max_retries=3):
    if model is None: model = MODEL_ID
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                response_format={"type": "json_object"},
                temperature=0.2
            )
            return json.loads(response.choices[0].message.content)
        except Exception as e:
            error_str = str(e).lower()
            if "429" in error_str or "rate limit" in error_str or "quota" in error_str:
                if attempt < max_retries - 1:
                    wait_time = (2 ** attempt) + random.uniform(0, 1)
                    print(f"⚠️  Rate limit hit, waiting {wait_time:.1f}s before retry {attempt + 1}/{max_retries}")
                    time.sleep(wait_time)
                    continue
                else:
                    raise e
            else:
                raise e
    raise Exception("Unexpected error")

print("✅ OpenAI client initialized")

In [ ]:
# ==== 7) Two-Pass Processing with Post-Processing AFTER Pass 2 ====
from tqdm import tqdm

def process_job_description(row_idx, row):
    job_text = f"""Position Summary: {row.get('Position Summary', '')}
Essential Functions: {row.get('Essential Functions', '')}
Work Experience: {row.get('Work Experience', '')}
Education: {row.get('Education', '')}
Licenses and Certifications: {row.get('Licenses and Certifications', '')}
Knowledge, Skills and Abilities: {row.get('Knowledge, Skills and Abilities', '')}"""
    job_title_original = row.get('Job Title', '')

    # === PASS 1: Initial Classification (raw LLM) ===
    pass1_prompt = f"""{zero_shot_prompt}
Available MNPS Roles: {', '.join(VALID_ROLES)}
{KSACS_TEXT}
Job Description to Classify:
{job_text}
**IMPORTANT**: 
- Ignore the job title completely
- Base classification solely on job attributes
- Use only approved MNPS roles and levels (I, II, III, Lead)
Return your response as a JSON object with:
{{
  "new_job_title": "...",
  "major_role_group": "...",
  "minor_sub_group": "...",
  "grouping_justification": "..."
}}"""
    
    try:
        pass1_response = call_llm_json_with_retry(pass1_prompt, MODEL_ID)
        raw_major = pass1_response.get('major_role_group', 'Other')
        raw_minor = pass1_response.get('minor_sub_group', 'I')
        justification = pass1_response.get('grouping_justification', 'No justification provided')
        raw_title = pass1_response.get('new_job_title', f"{raw_major} {raw_minor}")

        # === PASS 2: Self-Consistency (no post-processing yet) ===
        prior_output = {
            "major_role_group": raw_major,
            "minor_sub_group": raw_minor,
            "grouping_justification": justification
        }
        pass2_prompt = f"""{self_consistency_prompt}

**Your Previous Output:**
{json.dumps(prior_output, indent=2)}

**Now perform the self-consistency check and return the corrected (or unchanged) JSON. Include a reconstructed `new_job_title` that matches the corrected role and level.**
"""
        pass2_response = call_llm_json_with_retry(pass2_prompt, MODEL_ID)

        # Extract Pass 2 output
        major_role = pass2_response.get('major_role_group', raw_major)
        minor_role = pass2_response.get('minor_sub_group', raw_minor)
        new_job_title = pass2_response.get('new_job_title', f"{major_role} {minor_role}")
        justification = pass2_response.get('grouping_justification', justification)

        # === APPLY ALL POST-PROCESSING AFTER PASS 2 ===
        major_role = discourage_specialist(job_text, major_role)
        major_role = override_for_specialist_license(job_text, job_title_original, major_role)
        major_role = distinguish_supervisor_manager(job_text, major_role)
        major_role = refine_coordinator_coach_manager(job_text, major_role)
        major_role = downgrade_to_representative(job_text, job_title_original, major_role)
        major_role = promote_to_director_if_qualified(job_text, major_role)
        minor_role = normalize_minor(minor_role)
        minor_role = refine_coordinator_level(job_text, major_role, minor_role)
        minor_role = fix_executive_minor_sub_grouping(major_role, minor_role)
        minor_role = finalize_minor_role(major_role, minor_role)
        if not new_job_title or new_job_title == 'Unknown':
            if minor_role == '':
                new_job_title = f"{major_role}"
            else:
                new_job_title = f"{major_role} {minor_role}"

        return {
            'source_row_index': row_idx,
            'job_title_original': job_title_original,
            'new_job_title': new_job_title,
            'major_role_group': major_role,
            'minor_sub_group': minor_role,
            'grouping_justification': justification,
            'model_used': MODEL_ID
        }

    except Exception as e:
        print(f"Error processing row {row_idx}: {e}")
        return {
            'source_row_index': row_idx,
            'job_title_original': job_title_original,
            'new_job_title': 'Error',
            'major_role_group': 'Other',
            'minor_sub_group': 'I',
            'grouping_justification': f'Error: {str(e)}',
            'model_used': MODEL_ID
        }

# Process all
results = []
print("🚀 Starting two-pass batch processing with v7.5.5 fixes...")
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Processing jobs"):
    result = process_job_description(idx, row)
    results.append(result)
    time.sleep(0.2)

# Save
results_df = pd.DataFrame(results)
output_path = OUTPUTS_DIR / "Job_Classifications_Batch_gpt4o_v755_fixed.csv"
results_df.to_csv(output_path, index=False)
print(f"✅ Saved results to: {output_path}")

In [ ]:
# ==== 8) Summary Stats ====
preds = results_df.copy()
major_counts = preds['major_role_group'].value_counts()
minor_counts = preds['minor_sub_group'].value_counts()

summary_stats = pd.DataFrame({
    'metric': ['total_rows', 'unique_major_roles', 'unique_minor_roles', 'specialist_count', 'executive_lead_count'],
    'value': [
        len(preds),
        len(major_counts),
        len(minor_counts),
        int((preds['major_role_group'] == 'Specialist').sum()),
        int((preds['major_role_group'].isin(EXECUTIVE_ROLES) & (preds['minor_sub_group'] == 'Lead')).sum())
    ]
})
summary_stats.to_csv(OUTPUTS_DIR / "summary_stats_gpt4o_v755_fixed.csv", index=False)
print("📊 Summary stats generated")

In [ ]:
# ==== 9) Quality Check ====
alignment_issues = []
for idx, row in preds.iterrows():
    justification = str(row['grouping_justification']).lower()
    major_role = str(row['major_role_group']).lower()
    if major_role not in justification and major_role != 'other':
        alignment_issues.append({
            'row_index': row['source_row_index'],
            'major_role_group': row['major_role_group'],
            'justification_excerpt': row['grouping_justification'][:100] + '...'
        })

if alignment_issues:
    pd.DataFrame(alignment_issues).to_csv(OUTPUTS_DIR / "alignment_issues_gpt4o_v755_fixed.csv", index=False)
    print(f"⚠️  Found {len(alignment_issues)} alignment issues")
else:
    print("✅ No alignment issues found")

print("✅ Quality check completed")